In [5]:
import pickle
import pandas as pd

df = pd.read_pickle("my_dataframe.pkl")

ModuleNotFoundError: No module named 'numpy._core.numeric'

In [4]:
import joblib
joblib.dump(df, 'joblib_my_dataframe.pkl')


['joblib_my_dataframe.pkl']

In [17]:
import joblib
df = joblib.load('joblib_my_dataframe.pkl')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157786 entries, 0 to 157785
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Text              157786 non-null  object 
 1   emojis_present    157786 non-null  object 
 2   emojis            157786 non-null  object 
 3   anger             157786 non-null  float64
 4   disgust           157786 non-null  float64
 5   fear              157786 non-null  float64
 6   joy               157786 non-null  float64
 7   neutral           157786 non-null  float64
 8   sadness           157786 non-null  float64
 9   surprise          157786 non-null  float64
 10  negative          157786 non-null  float64
 11  neutral_sent      157786 non-null  float64
 12  positive          157786 non-null  float64
 13  emojis_vec_fixed  157786 non-null  object 
dtypes: float64(10), object(4)
memory usage: 16.9+ MB


In [4]:
print(type(df['emojis_vec_fixed'][0]))         
print(df['emojis_vec_fixed'].iloc[0].dtype)

<class 'numpy.ndarray'>
int64


In [6]:
import pickle
with open('emoji_dict.pickle', 'rb') as f:
    emoji_dict = pickle.load(f)

# Training Setup

In [1]:
import torch.nn as nn
from transformers import AutoModel
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [7]:
#subset of data for test 
#subset_df = df.head(100)

In [7]:
class BERTWithExtraFeatures(nn.Module):
    def __init__(self, num_labels, extra_feature_dim):
        super().__init__()
        self.bert = AutoModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size + extra_feature_dim, num_labels)

    def forward(self, input_ids, attention_mask, extra_features):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        combined = torch.cat((pooled_output, extra_features), dim=1)
        x = self.dropout(combined)
        return self.classifier(x)


In [8]:
class EmojiDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len, sentiment_cols, emotion_cols):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.sentiment_cols = sentiment_cols
        self.emotion_cols = emotion_cols

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        text = row['Text']
        labels = torch.tensor(row['emojis_vec_fixed'], dtype=torch.float) 
        
        # tokenize text
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        # for sentiment and emotion
        sentiment = torch.tensor(row[self.sentiment_cols].values.astype(float), dtype=torch.float)
        emotion = torch.tensor(row[self.emotion_cols].values.astype(float), dtype=torch.float)
        extra_features = torch.cat([sentiment, emotion], dim=0)

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'extra_features': extra_features,
            'labels': labels
        }


In [10]:
class BERTWithExtraFeatures(nn.Module):
    def __init__(self, num_labels, extra_feature_dim, pretrained_model='bert-base-uncased'):
        super(BERTWithExtraFeatures, self).__init__()
        self.bert = AutoModel.from_pretrained(pretrained_model)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size + extra_feature_dim, num_labels)

    def forward(self, input_ids, attention_mask, extra_features):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = bert_output.pooler_output 

        combined = torch.cat((pooled_output, extra_features), dim=1)  
        x = self.dropout(combined)
        logits = self.classifier(x)

        return logits


In [12]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
max_len = 128

#  sentiment and emotion col
sentiment_cols = ['positive', 'neutral_sent', 'negative']
emotion_cols = ['joy', 'sadness', 'anger','nuetral', 'fear', 'surprise', 'disgust']  

train_dataset = EmojiDataset(df, tokenizer, max_len, sentiment_cols, emotion_cols)

num_labels = len(df.iloc[0]['emojis_vec_fixed'])  # number of possible emojis
extra_feature_dim = len(sentiment_cols) + len(emotion_cols)

model = BERTWithExtraFeatures(num_labels=num_labels, extra_feature_dim=extra_feature_dim)

In [13]:
''''' below is testing on subset of dataa ''' '''
# Split the subset into training and validation sets
train_df, val_df = train_test_split(subset_df, test_size=0.1, random_state=42)

# Create datasets and dataloaders
train_dataset = EmojiDataset(train_df, tokenizer, max_len, sentiment_cols, emotion_cols)
val_dataset = EmojiDataset(val_df, tokenizer, max_len, sentiment_cols, emotion_cols)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16) '''
''''' above is testing on subset of dataa '''


# splittt
train_df, val_df = train_test_split(df, test_size=0.1)

#datasets
train_dataset = EmojiDataset(train_df, tokenizer, max_len, sentiment_cols, emotion_cols)
val_dataset = EmojiDataset(val_df, tokenizer, max_len, sentiment_cols, emotion_cols)

#dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# GPU 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Define loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [14]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in tqdm(dataloader):
        try:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            extra_features = batch['extra_features'].to(device)
            labels = batch['labels'].to(device)
    
            optimizer.zero_grad()
    
            outputs = model(input_ids, attention_mask, extra_features)
            loss = criterion(outputs, labels)
    
            loss.backward()
            optimizer.step()
    
            total_loss += loss.item()
        except Exception as e:
            print(f"Error encountered during training: {e}")
            continue

    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in dataloader:
            try:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                extra_features = batch['extra_features'].to(device)
                labels = batch['labels'].to(device)
    
                outputs = model(input_ids, attention_mask, extra_features)
                loss = criterion(outputs, labels)
                total_loss += loss.item()
            except Exception as e: 
                print(f"Error encountered during evaluation: {e}")
                continue

    return total_loss / len(dataloader)

In [ ]:
final_model_path = 'official_full_model.pth'
num_epochs = 6

for epoch in range(num_epochs):
    print(f'Epoch {epoch + 1}/{num_epochs}')

    try:
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        print(f'Train Loss: {train_loss:.4f}')
    except Exception as e:
        print(f"Error during training at epoch{epoch + 1}: {e}")
        continue
    #validation
    try:
        val_loss = evaluate(model, val_loader, criterion, device)
        print(f'Val Loss: {val_loss:.4f}')
    except Exception as e:
        print(f"Error during evaluation at epoch {epoch + 1}: {e}")
        continue
    


### Training done on Colab

In [ ]:
torch.save(model.state_dict(), '1bert_with_features.pth')
torch.save(model, final_model_path)
torch.save(model.state_dict(), "model_w.pth")

## Evaluate (done on colab)

In [ ]:
import torch
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
from tqdm import tqdm

# Function to get predictions from the fine-tuned model
def get_predictions(model, text_data, tokenizer, device, extra_features_data, threshold=0.5):
    model.eval()
    all_predictions = []

    with torch.no_grad():
        for text, extra_features in tqdm(zip(text_data, extra_features_data)):
            #tokenize
            inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=512)

            if 'token_type_ids' in inputs:
                del inputs['token_type_ids']  

            inputs = {key: value.to(device) for key, value in inputs.items()}

            #  extra_features to torch tensor and move to 'device'
            extra_features = torch.tensor(extra_features, dtype=torch.float).to(device)

            # reshape as necessaryy
            extra_features = extra_features.unsqueeze(0) if extra_features.dim() == 1 else extra_features

            # this is "raw output"
            logits = model(**inputs, extra_features=extra_features)  # Include extra_features in the forward pass

            # get probs
            probs = torch.sigmoid(logits)

            # binarize predictions based on the threshold
            preds = (probs > threshold).int()
            all_predictions.append(preds.cpu())

    # concatenate all predictions 
    model_predictions = torch.cat(all_predictions)

    return model_predictions

# Extract the text and actual emojis (ground truth) from the dataframe
text_data = df['Text'].tolist()  # The text column in your dataframe
true_labels = np.array(df['emojis_vec_fixed'].tolist())  # Ground truth emojis

# Extract the extra features (emotion, sentiment, etc.)
extra_features_data = np.array(df[['joy', 'sadness', 'anger','neutral', 'fear', 'surprise', 'disgust','positive', 'neutral_sent', 'negative']])  # Adjust to your actual extra features columns

model_predictions = get_predictions(model, text_data, tokenizer, device, extra_features_data)

precision = precision_score(true_labels, model_predictions, average='micro')
recall = recall_score(true_labels, model_predictions, average='micro')
f1 = f1_score(true_labels, model_predictions, average='micro')

# Print metrics
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
